In [34]:
# Import tabular data tools and the Hugging Face dataset loader.
import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from sklearn.model_selection import train_test_split

In [ ]:
# Load the source CSV from the Hugging Face Hub into a DatasetDict.
dataset = load_dataset("csv", data_files="hf://datasets/motidev/wellness-tourism-dataset/raw/tourism.csv")

In [ ]:
# Inspect the available dataset splits and their features.
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome'],
        num_rows: 4128
    })
})

In [ ]:
# Convert the training split to pandas for exploratory analysis.
df = dataset["train"].to_pandas()

# Preview sample records and confirm the table loaded as expected.
df.head()

,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [ ]:
# Check the number of records and features.
df.shape

(4128, 21)

In [ ]:
# List feature names before selecting or transforming columns.
df.columns.to_list()

['Unnamed: 0',
 'CustomerID',
 'ProdTaken',
 'Age',
 'TypeofContact',
 'CityTier',
 'DurationOfPitch',
 'Occupation',
 'Gender',
 'NumberOfPersonVisiting',
 'NumberOfFollowups',
 'ProductPitched',
 'PreferredPropertyStar',
 'MaritalStatus',
 'NumberOfTrips',
 'Passport',
 'PitchSatisfactionScore',
 'OwnCar',
 'NumberOfChildrenVisiting',
 'Designation',
 'MonthlyIncome']

In [ ]:
# Review inferred data types to identify columns needing conversion.
df.dtypes

Unnamed: 0                    int64
CustomerID                    int64
ProdTaken                     int64
Age                         float64
TypeofContact                   str
CityTier                      int64
DurationOfPitch             float64
Occupation                      str
Gender                          str
NumberOfPersonVisiting        int64
NumberOfFollowups           float64
ProductPitched                  str
PreferredPropertyStar       float64
MaritalStatus                   str
NumberOfTrips               float64
Passport                      int64
PitchSatisfactionScore        int64
OwnCar                        int64
NumberOfChildrenVisiting    float64
Designation                     str
MonthlyIncome               float64
dtype: object

In [ ]:
# Count missing values per feature to guide imputation or removal.
df.isnull().sum()

Unnamed: 0                  0
CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [ ]:
# Detect fully duplicated records that could bias analysis or training.
df.duplicated().sum()

np.int64(0)

In [ ]:
# Examine the target distribution for potential class imbalance.
df["ProdTaken"].value_counts()

ProdTaken
0    3331
1     797
Name: count, dtype: int64

In [ ]:
# Summarize column types, non-null counts, and memory usage.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4128 entries, 0 to 4127
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                4128 non-null   int64  
 1   CustomerID                4128 non-null   int64  
 2   ProdTaken                 4128 non-null   int64  
 3   Age                       4128 non-null   float64
 4   TypeofContact             4128 non-null   str    
 5   CityTier                  4128 non-null   int64  
 6   DurationOfPitch           4128 non-null   float64
 7   Occupation                4128 non-null   str    
 8   Gender                    4128 non-null   str    
 9   NumberOfPersonVisiting    4128 non-null   int64  
 10  NumberOfFollowups         4128 non-null   float64
 11  ProductPitched            4128 non-null   str    
 12  PreferredPropertyStar     4128 non-null   float64
 13  MaritalStatus             4128 non-null   str    
 14  NumberOfTrips      

## Data Cleaning

The dataset was inspected for missing values, duplicate records, and unnecessary columns.

- No missing values were identified.
- No duplicate records were identified.
- `Unnamed: 0` was removed because it represents an unnecessary index column.
- `CustomerID` was removed because it is a unique customer identifier and does not provide meaningful predictive information for the model.

In [29]:
columns_to_drop = ["Unnamed: 0", "CustomerID"]
df_clean = df.drop(columns=columns_to_drop)

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

df_clean.head()

Original shape: (4128, 21)
Cleaned shape: (4128, 19)


,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,32.0,Company Invited,1,8.0,Salaried,Male,3,3.0,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [24]:
# Separating features and target variable
X = df_clean.drop(columns=["ProdTaken"])
y = df_clean["ProdTaken"]
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (4128, 18)
Target shape: (4128,)


In [27]:
#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training features shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Testing features shape:", X_test.shape)
print("Testing target shape:", y_test.shape)

Training features shape: (3302, 18)
Training target shape: (3302,)
Testing features shape: (826, 18)
Testing target shape: (826,)


In [28]:
# Verifying stratification
print("Overall")
print(y.value_counts(normalize=True))

print("\nTraining")
print(y_train.value_counts(normalize=True))
print("\nTesting")
print(y_test.value_counts(normalize=True))

Overall
ProdTaken
0    0.806928
1    0.193072
Name: proportion, dtype: float64

Training
ProdTaken
0    0.806784
1    0.193216
Name: proportion, dtype: float64

Testing
ProdTaken
0    0.807506
1    0.192494
Name: proportion, dtype: float64


In [30]:
# Combine features and target for training and testing datasets

train_df = X_train.copy()
train_df["ProdTaken"] = y_train

test_df = X_test.copy()
test_df["ProdTaken"] = y_test

print("Training dataset shape:", train_df.shape)
print("Testing dataset shape:", test_df.shape)

Training dataset shape: (3302, 19)
Testing dataset shape: (826, 19)


In [31]:
train_df.head()

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome,ProdTaken
3187,55.0,Self Enquiry,1,17.0,Small Business,Female,4,4.0,Deluxe,5.0,Unmarried,8.0,1,1,0,1.0,Manager,23118.0,0
1933,39.0,Self Enquiry,1,9.0,Salaried,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,4,0,2.0,Executive,22622.0,0
695,42.0,Company Invited,2,8.0,Small Business,Male,3,1.0,Deluxe,5.0,Divorced,1.0,0,2,0,2.0,Manager,21272.0,0
1949,37.0,Self Enquiry,1,12.0,Salaried,Female,3,5.0,Basic,5.0,Divorced,2.0,1,2,1,1.0,Executive,98678.0,0
2485,23.0,Self Enquiry,1,7.0,Salaried,Male,3,5.0,Deluxe,3.0,Divorced,8.0,0,2,1,1.0,Manager,23453.0,0


In [33]:
os.makedirs("../data", exist_ok=True)

train_df.to_csv("../data/train.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

print("Training and testing datasets have been saved to the '../data' directory.")

Training and testing datasets have been saved to the '../data' directory.


## Upload Processed Data to Hugging Face

The cleaned dataset was split into training and testing sets using a stratified
80/20 split. The resulting datasets are uploaded to the Hugging Face Dataset
repository so that subsequent model-training stages can retrieve the processed
data directly from the repository.

In [36]:
api = HfApi()

repo_id = "motidev/wellness-tourism-dataset"


In [38]:
api.upload_file(
    path_or_fileobj="../data/train.csv",
    path_in_repo="processed/train.csv",
    repo_id=repo_id,
    repo_type="dataset"
)

print("File uploaded successfully.")

File uploaded successfully.


In [39]:
api.upload_file(
    path_or_fileobj="../data/test.csv",
    path_in_repo="processed/test.csv",
    repo_id=repo_id,
    repo_type="dataset"
)

print("Test file uploaded successfully.")

Test file uploaded successfully.


In [40]:
# Verify that the files have been uploaded successfully

processed_dataset = load_dataset(
    "csv",
    data_files={
        "train": f"hf://datasets/{repo_id}/processed/train.csv",
        "test": f"hf://datasets/{repo_id}/processed/test.csv"
    }
)

processed_dataset

Generating train split: 3302 examples [00:00, 235373.15 examples/s]
Generating test split: 826 examples [00:00, 178361.57 examples/s]


DatasetDict({
    train: Dataset({
        features: ['Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome', 'ProdTaken'],
        num_rows: 3302
    })
    test: Dataset({
        features: ['Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome', 'ProdTaken'],
        num_rows: 826
    })
})